In [19]:
import zipfile, os

zip_file_path = '/content/drive-download-20260612T140644Z-3-001.zip'
extract_dir = '/content/datasets/'

# Create the extraction directory if it doesn't exist
os.makedirs(extract_dir, exist_ok=True)

if os.path.exists(zip_file_path):
    print(f"Extracting {zip_file_path} to {extract_dir}...")
    with zipfile.ZipFile(zip_file_path, 'r') as z:
        z.extractall(extract_dir)
    print("Extraction complete.")
    print(f"Contents of {extract_dir}:")
    print(os.listdir(extract_dir))
else:
    print(f"Error: Zip file not found at {zip_file_path}.")
    print("Please ensure the zip file is uploaded or correctly mounted.")

Extracting /content/drive-download-20260612T140644Z-3-001.zip to /content/datasets/...
Extraction complete.
Contents of /content/datasets/:
['SYNTH.npy', 'MP8.npy', 'MP50.npy', 'Rowing.npy', 'CMJ.npy']


In [20]:
!git clone https://github.com/mlgig/FM4MTSC.git
!pip install momentfm --no-deps -q
!pip install einops -q

fatal: destination path 'FM4MTSC' already exists and is not an empty directory.


In [21]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import Adam
from sklearn.preprocessing import LabelEncoder
from momentfm import MOMENTPipeline

In [22]:
data = np.load('/content/datasets/SYNTH.npy', allow_pickle=True)
print(type(data))
print(data.shape if hasattr(data, 'shape') else 'no shape')
if hasattr(data, 'item'):
    d = data.item()
    print(type(d))
    print(d.keys() if hasattr(d, 'keys') else d)

<class 'numpy.ndarray'>
()
<class 'dict'>
dict_keys(['train', 'test'])


In [23]:
datasets = {}
for name in ['SYNTH', 'CMJ', 'MP8', 'MP50']:
    data = np.load(f'/content/datasets/{name}.npy', allow_pickle=True).item()
    print(f'{name} train keys:', data['train'].keys())
    break

SYNTH train keys: dict_keys(['X', 'y', 'ground_truth', 'additional_info'])


In [24]:
datasets = {}
for name in ['SYNTH', 'CMJ', 'MP8', 'MP50']:
    data = np.load(f'/content/datasets/{name}.npy', allow_pickle=True).item()
    X_tr = torch.tensor(data['train']['X'], dtype=torch.float32)
    X_te = torch.tensor(data['test']['X'], dtype=torch.float32)
    le = LabelEncoder()
    y_tr = torch.tensor(le.fit_transform(data['train']['y']), dtype=torch.long)
    y_te = torch.tensor(le.transform(data['test']['y']), dtype=torch.long)
    datasets[name] = (X_tr, X_te, y_tr, y_te, le)
    print(f'{name}: X_train={X_tr.shape}, classes={le.classes_}')

SYNTH: X_train=torch.Size([7500, 8, 500]), classes=['0' '1']
CMJ: X_train=torch.Size([419, 3, 384]), classes=['0' '1' '2']
MP8: X_train=torch.Size([1426, 8, 161]), classes=[0 1 2 3]
MP50: X_train=torch.Size([1426, 50, 161]), classes=[0 1 2 3]


In [25]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using:', device)

results = {}

Using: cuda


In [26]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [27]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using:', device)

results = {}

for name in ['MP8', 'MP50']:
    print(f'\n=== {name} ===')
    X_tr, X_te, y_tr, y_te, le = datasets[name]

    model = MOMENTPipeline.from_pretrained(
        'AutonLab/MOMENT-1-large',
        model_kwargs={'task_name': 'classification', 'n_channels': X_tr.shape[1], 'num_class': len(le.classes_)}
    )
    model.init()
    model = model.to(device)

    for param in model.parameters():
        param.requires_grad = False
    for param in model.head.parameters():
        param.requires_grad = True

    loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=32, shuffle=True)
    optimizer = Adam(model.head.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    model.train()
    for epoch in range(2):
        total_loss = 0
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            out = model(x_enc=X_batch)
            loss = criterion(out.logits, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f'Epoch {epoch+1} loss: {total_loss/len(loader):.4f}')

    torch.save(model.state_dict(), f'/content/drive/MyDrive/MOMENT_{name}_trained.pt')
    print('Saved!')

    model.eval()
    all_preds = []
    with torch.no_grad():
        for i in range(0, len(X_te), 32):
            X_batch = X_te[i:i+32].to(device)
            out = model(x_enc=X_batch)
            preds = out.logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)

    acc = accuracy_score(y_te.numpy(), all_preds)
    results[name] = acc
    print(f'Accuracy: {acc:.4f}')
    print(classification_report(y_te.numpy(), all_preds))

print('\n=== SUMMARY ===')
for name, acc in results.items():
    print(f'{name}: {acc:.4f}')

Using: cuda

=== MP8 ===


/usr/local/lib/python3.12/dist-packages/momentfm/models/moment.py:174: UserWarning: Only reconstruction head is pre-trained. Classification and forecasting heads must be fine-tuned.
  warnings.warn("Only reconstruction head is pre-trained. Classification and forecasting heads must be fine-tuned.")


Epoch 1 loss: 1.3421
Epoch 2 loss: 1.2244
Saved!


NameError: name 'accuracy_score' is not defined

In [11]:
from sklearn.metrics import accuracy_score, classification_report

name = 'SYNTH'
X_tr, X_te, y_tr, y_te, le = datasets[name]

model = MOMENTPipeline.from_pretrained(
    'AutonLab/MOMENT-1-large',
    model_kwargs={'task_name': 'classification', 'n_channels': X_tr.shape[1], 'num_class': len(le.classes_)}
)
model.init()
model = model.to(device)

for param in model.parameters():
    param.requires_grad = False
for param in model.head.parameters():
    param.requires_grad = True

loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=32, shuffle=True)
optimizer = Adam(model.head.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

model.train()
for epoch in range(2):
    total_loss = 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        out = model(x_enc=X_batch)
        loss = criterion(out.logits, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'Epoch {epoch+1} loss: {total_loss/len(loader):.4f}')

torch.save(model.state_dict(), '/content/drive/MyDrive/MOMENT_SYNTH_trained.pt')
print('Saved!')

model.eval()
all_preds = []
with torch.no_grad():
    for i in range(0, len(X_te), 32):
        X_batch = X_te[i:i+32].to(device)
        out = model(x_enc=X_batch)
        preds = out.logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)

print(accuracy_score(y_te.numpy(), all_preds))
print(classification_report(y_te.numpy(), all_preds))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/951 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.39G [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/momentfm/models/moment.py:174: UserWarning: Only reconstruction head is pre-trained. Classification and forecasting heads must be fine-tuned.
  warnings.warn("Only reconstruction head is pre-trained. Classification and forecasting heads must be fine-tuned.")


Epoch 1 loss: 0.5613
Epoch 2 loss: 0.4618
Saved!
0.815
              precision    recall  f1-score   support

           0       0.92      0.69      0.79       500
           1       0.75      0.94      0.83       500

    accuracy                           0.81      1000
   macro avg       0.83      0.81      0.81      1000
weighted avg       0.83      0.81      0.81      1000



In [12]:
from sklearn.metrics import accuracy_score, classification_report

name = 'CMJ'
X_tr, X_te, y_tr, y_te, le = datasets[name]

model = MOMENTPipeline.from_pretrained(
    'AutonLab/MOMENT-1-large',
    model_kwargs={'task_name': 'classification', 'n_channels': X_tr.shape[1], 'num_class': len(le.classes_)}
)
model.init()
model = model.to(device)

for param in model.parameters():
    param.requires_grad = False
for param in model.head.parameters():
    param.requires_grad = True

loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=32, shuffle=True)
optimizer = Adam(model.head.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

model.train()
for epoch in range(2):
    total_loss = 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        out = model(x_enc=X_batch)
        loss = criterion(out.logits, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'Epoch {epoch+1} loss: {total_loss/len(loader):.4f}')

torch.save(model.state_dict(), '/content/drive/MyDrive/MOMENT_CMJ_trained.pt')
print('Saved!')

model.eval()
all_preds = []
with torch.no_grad():
    for i in range(0, len(X_te), 32):
        X_batch = X_te[i:i+32].to(device)
        out = model(x_enc=X_batch)
        preds = out.logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)

print(accuracy_score(y_te.numpy(), all_preds))
print(classification_report(y_te.numpy(), all_preds))

/usr/local/lib/python3.12/dist-packages/momentfm/models/moment.py:174: UserWarning: Only reconstruction head is pre-trained. Classification and forecasting heads must be fine-tuned.
  warnings.warn("Only reconstruction head is pre-trained. Classification and forecasting heads must be fine-tuned.")


Epoch 1 loss: 1.0527
Epoch 2 loss: 0.9338
Saved!
0.9106145251396648
              precision    recall  f1-score   support

           0       0.89      0.93      0.91        60
           1       0.98      0.83      0.90        59
           2       0.88      0.97      0.92        60

    accuracy                           0.91       179
   macro avg       0.92      0.91      0.91       179
weighted avg       0.92      0.91      0.91       179

